In [ ]:
%load_ext cudf.pandas

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
#
import numpy as np  # linear algebra
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
    import pandas
else:
    import pandas as pd
import time

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("../input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing libs
import numpy as np
import warnings

warnings.filterwarnings("ignore")
# import plotly as py
# import plotly.graph_objs as go
import os

# py.offline.init_notebook_mode(connected = True)
# print(os.listdir("../input"))
import datetime as dt

In [ ]:
%%time
### cell 0 ###

df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/input/netflix-shows/netflix_titles.csv"
)
factor = 500
df = pd.concat([df] * factor, ignore_index=True)
df.head(3)

In [ ]:
%%time
### cell 1 ###

# Missing data

for i in df.columns:
    null_rate = df[i].isna().sum() / len(df) * 100
    if null_rate > 0:
        print("{} null rate: {}%".format(i, round(null_rate, 2)))

In [ ]:
%%time
### cell 2 ###

# Replacments

df["country"] = df["country"].fillna(df["country"].mode()[0])


df["cast"].replace(np.nan, "No Data", inplace=True)
df["director"].replace(np.nan, "No Data", inplace=True)

# Drops

df.dropna(inplace=True)

# Drop Duplicates

In [ ]:
%%time
### cell 3 ###

df.isnull().sum()

In [ ]:
%%time
### cell 4 ###

df.info()

In [ ]:
%%time
### cell 5 ###

df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")


df["month_added"] = df["date_added"].dt.month
df["index"] = df["date_added"].dt.month_name()
df["year_added"] = df["date_added"].dt.year

df.head(3)

In [ ]:
%%time
### cell 6 ###

# For viz: Ratio of Movies & TV shows

x = df.groupby(["type"])["type"].count()
y = len(df)
r = (x / y).round(2)

mf_ratio = pd.DataFrame(r).T

In [ ]:
%%time
### cell 7 ###

for i_1 in mf_ratio.index:
    _ = int(mf_ratio["Movie"][i_1] * 100)
    _ = mf_ratio["Movie"][i_1] / 2
    _ = mf_ratio["Movie"][i_1] / 2

for i_2 in mf_ratio.index:
    _ = int(mf_ratio["TV Show"][i_2] * 100)
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2

In [ ]:
%%time
### cell 8 ###

df["count"] = 1

# Extract the first country using GPU-accelerated split with expand to avoid .str.get CPU calls
# We split once (n=1) on the first comma and take the 0th column
df["first_country"] = df["country"].str.split(",", n=1, expand=True)[0]

# Shorten common country names in one GPU replace
df["first_country"] = df["first_country"].replace(
    {"United States": "USA", "United Kingdom": "UK", "South Korea": "S. Korea"}
)

# Map ratings to age groups with GPU replace
ratings_ages = {
    "TV-PG": "Older Kids",
    "TV-MA": "Adults",
    "TV-Y7-FV": "Older Kids",
    "TV-Y7": "Older Kids",
    "TV-14": "Teens",
    "R": "Adults",
    "TV-Y": "Kids",
    "NR": "Adults",
    "PG-13": "Teens",
    "TV-G": "Kids",
    "PG": "Older Kids",
    "G": "Kids",
    "UR": "Adults",
    "NC-17": "Adults",
}

df["target_ages"] = df["rating"].replace(ratings_ages)

# Clean up the genre strings in one GPU regex replace and split into lists
# This collapses any spaces around commas into a single comma
df["genre"] = df["listed_in"].str.replace(r"\s*,\s*", ",", regex=True).str.split(",")

In [ ]:
%%time
### cell 9 ###

data = df.groupby("first_country")["count"].sum().sort_values(ascending=False)[:10]

In [ ]:
%%time
### cell 10 ###

country_order = df["first_country"].value_counts()[:11].index

# Use GPU‐friendly pivot instead of CPU‐bound value_counts + unstack
counts = df.groupby(["first_country", "type"]).size().reset_index(name="count")
data_q2q3 = (
    counts.pivot(index="first_country", columns="type", values="count")
    .fillna(0)
    .loc[country_order]
)

# compute total and ratios
data_q2q3["sum"] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    data_q2q3[["Movie", "TV Show"]]
    .div(data_q2q3["sum"], axis=0)
    .sort_values(by="Movie", ascending=False)
    .iloc[::-1]
)

In [ ]:
%%time

### cell 11 ###

order = pd.DataFrame(
    df.groupby("rating")["count"].sum().sort_values(ascending=False).reset_index()
)
rating_order = list(order["rating"])

In [ ]:
%%time
### cell 12 ###


rating_dummies = pd.get_dummies(df["rating"])[rating_order]
mf = rating_dummies.groupby(df["type"]).sum().sort_index().astype(int)
movie = mf.loc["Movie"]
tv = -mf.loc["TV Show"]

In [ ]:
%%time
### cell 13 ###

# Compute exactly the same "data_sub" as the original, but entirely on the GPU:
data_sub = (
    df.dropna(
        subset=["year_added"]
    )  # drop any rows with missing year_added (value_counts would skip these)
    .groupby(["type", "year_added"])  # GPU groupby on type & year
    .size()  # count entries per (type, year)
    .unstack(level="year_added")  # pivot years into columns
    .fillna(0)  # fill missing type–year combos with 0
    .loc[["TV Show", "Movie"]]  # ensure the row order matches the original
    .cumsum(axis=0)  # cumulative sum down the two rows (for stacking)
    .T  # transpose → index=year, columns=type
)

In [ ]:
%%time
### cell 14 ###

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

# Use a cuDF-backed CategoricalDtype instead of pd.Categorical (CPU) to keep work on the GPU
cat_dtype = pd.CategoricalDtype(categories=month_order, ordered=True)
df["index"] = df["index"].astype(cat_dtype)

In [ ]:
%%time
### cell 15 ###

# Optimized for cudf: use GPU-accelerated groupby.size() instead of value_counts()
# and pass fill_value to unstack to avoid a separate fillna call.
data_sub = (
    df.groupby(["type", "index"])
    .size()
    .unstack(fill_value=0)
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)

In [ ]:
%%time

### cell 16 ###

data_sub2 = data_sub

data_sub2["Value"] = data_sub2["Movie"] + data_sub2["TV Show"]
data_sub2 = data_sub2.reset_index()

df_polar = data_sub2.sort_values(by="index", ascending=False)


color_map = ["#221f1f" for _ in range(12)]
color_map[0] = color_map[11] = "#b20710"  # color highlight

In [ ]:
%%time

### cell 17 ###

# Genres
from sklearn.preprocessing import MultiLabelBinarizer

import matplotlib.colors


# Custom colour map based on Netflix palette
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    "", ["#221f1f", "#b20710", "#f5f5f1"]
)

In [ ]:
%%time


### cell 18 ###
def genre_heatmap(df, title):
    # Vectorized cleaning and splitting of the "listed_in" column
    df["genre"] = (
        df["listed_in"]
        .str.replace(" ,", ",", regex=False)
        .str.replace(", ", ",", regex=False)
        .str.split(",")
    )
    # Explode the lists to get a flat column of all genres
    all_genres = df["genre"].explode()
    # Compute the unique genres directly on the GPU
    unique_genres = all_genres.unique()
    print(
        "There are {} types in the Netflix {} Dataset".format(len(unique_genres), title)
    )
    test_1 = df["genre"]


# Filter once on the GPU
df_tv = df[df["type"] == "TV Show"]
df_movies = df[df["type"] == "Movie"]

# Invoke the optimized function
genre_heatmap(df_movies, "Movie")

In [ ]:
%%time

### cell 19 ###


data = (
    df.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data = data["first_country"]


df_heatmap = df.loc[df["first_country"].isin(data)]

In [ ]:
%%time
### cell 20 ###

# Replace CPU-bound crosstab with GPU groupby + unstack + vectorized normalization
counts = (
    df_heatmap.groupby(["first_country", "target_ages"]).size().unstack(fill_value=0)
)
# normalize each row (index) and transpose
df_heatmap = counts.div(counts.sum(axis=1), axis=0).T

In [ ]:
%%time
### cell 21 ###
# Display existing DataFrames (no change needed)
df_movies
df_tv

# Compute top 10 countries by total movie count using a GPU-friendly nlargest
# and extract just the country names as a Series
data = (
    df_movies.groupby("first_country")["count"]
    .sum()
    .nlargest(10)
    .reset_index(name="count")["first_country"]
)

# Filter to only those top countries
df_loli = df_movies[df_movies["first_country"].isin(data)]

# Compute the mean release and added years, then round
loli = df_loli.groupby("first_country")[["release_year", "year_added"]].mean().round()

# Prepare the sorted views for plotting
ordered_df = loli.sort_values("release_year")
ordered_df_rev = loli.sort_values("release_year", ascending=False)

In [ ]:
%%time

### cell 22 ###

data = (
    df_tv.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data = data["first_country"]
df_loli = df_tv.loc[df_tv["first_country"].isin(data)]

loli = df_loli.groupby("first_country")[["release_year", "year_added"]].mean().round()


# Reorder it following the values of the first value:
ordered_df = loli.sort_values(by="release_year")

ordered_df_rev = loli.sort_values(by="release_year", ascending=False)

In [ ]:
%%time
### cell 23 ###

us_ind = df[(df["first_country"] == "USA") | (df["first_country"] == "India")]

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = df._to_pandas()

data_sub = (
    df.groupby("first_country")["year_added"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["USA", "India"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = pd.DataFrame(df)
    data_sub = pd.DataFrame(data_sub)

In [ ]:
%%time
### cell 24 ###

# Filter for USA and India (GPU)
us_ind = df[(df["first_country"] == "USA") | (df["first_country"] == "India")]

# Create indicator columns for each country (GPU)
us_ind["USA"] = (us_ind["first_country"] == "USA").astype("int32")
us_ind["India"] = (us_ind["first_country"] == "India").astype("int32")

# Aggregate yearly counts per country (GPU)
yearly_counts = us_ind.groupby("year_added")[["USA", "India"]].sum()

# Compute half of the total counts per year for centering (GPU)
half_total = (yearly_counts["USA"] + yearly_counts["India"]) / 2

# Build data_sub with baseline and lobe positions (GPU)
data_sub = yearly_counts.copy()
data_sub.insert(0, "base", -half_total)
data_sub["USA"] = yearly_counts["USA"] - half_total
data_sub["India"] = half_total

In [ ]:
%%time

### cell 25 ###

# STEFANOS: Uneeded
# from wordcloud import WordCloud
import random
from PIL import Image
import matplotlib

# Custom colour map based on Netflix palette
# STEFANOS: Disable plotting
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#221f1f', '#b20710'])

text = df["title"].str.replace(r"[,\.\[\]']", "", regex=True).str.cat(sep=" ")


# STEFANOS: Can't find the icon
# mask = np.array(Image.open('../input/netflix-icon-new/f6974e017d3f6196c4cbe284ee3eaf4e.png'))


# STEFANOS: Disable ML and plotting
# wordcloud = WordCloud(background_color = 'white', width = 500,  height = 200,colormap=cmap, max_words = 150, mask = mask).generate(text)

# plt.figure( figsize=(5,5))
# plt.imshow(wordcloud, interpolation = 'bilinear')
# plt.axis('off')
# plt.tight_layout(pad=0)
# plt.show()